# SASRec Eval-Seed Result Tables

This notebook mirrors the structure of `notebooks/eval_seed_tables.ipynb`, but reads SASRec evaluation artifacts produced by `scripts/sasrec_eval.py`.

It focuses on the seeded evaluation protocol (`eval_mode == "eval_seeds"`) so the comparison is like-for-like with the RPG notebook and with the paper-style reporting tables.

Paper reference values below were extracted from Table 2 of `docs/original_paper/2506.05781v1.pdf` and verified against both:
- the parsed text file `docs/original_paper/2506.05781v1.txt`
- direct visual inspection of the rendered PDF page containing Table 2


In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

ROOT = Path.cwd().resolve()
while ROOT.name not in {"RPG", "RPG-uva"} and ROOT.parent != ROOT:
    ROOT = ROOT.parent

HOME_ARTIFACT_ROOT = ROOT / "artifacts" / "sasrec" / "eval_seeds"
GROUP_ARTIFACT_ROOT = Path("/projects/prjs2120/groups/group_16/artifacts/sasrec/eval_seeds")
ARTIFACT_ROOTS: list[Path] = []
for candidate in (GROUP_ARTIFACT_ROOT, HOME_ARTIFACT_ROOT):
    if candidate not in ARTIFACT_ROOTS:
        ARTIFACT_ROOTS.append(candidate)

REPORT_METRIC = "ndcg@10"
TABLE_METRICS = ["recall@5", "ndcg@5", "recall@10", "ndcg@10"]
METRIC_LABELS = {"recall@5": "R@5", "ndcg@5": "N@5", "recall@10": "R@10", "ndcg@10": "N@10"}
RELATIVE_EQ_MARGIN = 0.05
TOST_CI_LEVEL = 0.90
BOOTSTRAP_SAMPLES = 5000
BOOTSTRAP_SEED = 2024
REQUIRED_EVAL_MODE = "eval_seeds"

# Verified from Table 2 in docs/original_paper/2506.05781v1.pdf (page 5) and cross-checked
# against docs/original_paper/2506.05781v1.txt.
PAPER_VALUES = {
    "recall@5": {
        "Sports": 0.0233,
        "Beauty": 0.0387,
        "Toys": 0.0463,
        "CDs": 0.0351,
    },
    "ndcg@5": {
        "Sports": 0.0154,
        "Beauty": 0.0249,
        "Toys": 0.0306,
        "CDs": 0.0177,
    },
    "recall@10": {
        "Sports": 0.0350,
        "Beauty": 0.0605,
        "Toys": 0.0675,
        "CDs": 0.0619,
    },
    "ndcg@10": {
        "Sports": 0.0192,
        "Beauty": 0.0318,
        "Toys": 0.0374,
        "CDs": 0.0263,
    },
}


def _format_scalar_for_latex(value: Any, formatter: Any, na_rep: str) -> Any:
    if pd.isna(value):
        return na_rep
    if formatter is None:
        return value
    if callable(formatter):
        return formatter(value)
    return formatter.format(value)


def latex_ready_frame(
    df: pd.DataFrame,
    formatter: Any = None,
    column_formatters: dict | None = None,
    na_rep: str = "",
) -> pd.DataFrame:
    formatted = df.copy()
    if formatter is not None:
        formatted = formatted.applymap(
            lambda value: _format_scalar_for_latex(value, formatter, na_rep)
        )
    if column_formatters:
        for column, column_formatter in column_formatters.items():
            formatted[column] = formatted[column].map(
                lambda value: _format_scalar_for_latex(value, column_formatter, na_rep)
            )
    return formatted


def emit_latex_table(
    name: str,
    df: pd.DataFrame,
    formatter: Any = None,
    column_formatters: dict | None = None,
    na_rep: str = "",
    escape: bool = True,
    index: bool = True,
) -> None:
    latex_df = latex_ready_frame(
        df,
        formatter=formatter,
        column_formatters=column_formatters,
        na_rep=na_rep,
    )
    print(f"\n{name} LaTeX:\n")
    print(
        latex_df.to_latex(
            index=index,
            escape=escape,
            na_rep=na_rep,
            multicolumn=True,
            multirow=True,
        )
    )

ARTIFACT_ROOTS


[PosixPath('/projects/prjs2120/groups/group_16/artifacts/sasrec/eval_seeds'),
 PosixPath('/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval_seeds')]

In [2]:
TRACK_LABELS = {
    "released_readme": "released README",
    "paper_appendix": "paper appendix",
}

DATASET_LABELS = {
    "Sports_and_Outdoors": "Sports",
    "sports_and_outdoors": "Sports",
    "Beauty": "Beauty",
    "beauty": "Beauty",
    "Toys_and_Games": "Toys",
    "toys_and_games": "Toys",
    "CDs_and_Vinyl": "CDs",
    "cds_and_vinyl": "CDs",
}


def _metric_summary(payload: dict, metric: str) -> dict:
    for row in payload.get("metric_summary", []):
        if row.get("metric") == metric:
            return row
    raise KeyError(f"Metric {metric!r} not found in metric_summary")


def _paper_value(dataset: str, metric: str) -> float | None:
    value = PAPER_VALUES.get(metric, {}).get(dataset)
    return None if value is None else float(value)


def _mean_visited_items(payload: dict) -> float:
    values = [row.get("n_visited_items") for row in payload.get("per_seed_summary", [])]
    values = [float(value) for value in values if value is not None]
    return float(np.mean(values)) if values else math.nan


def _artifact_root_for(summary_path: Path) -> Path | None:
    for artifact_root in ARTIFACT_ROOTS:
        try:
            summary_path.relative_to(artifact_root)
            return artifact_root
        except ValueError:
            continue
    return None


def _track_and_dataset(summary_path: Path) -> tuple[str, str]:
    artifact_root = _artifact_root_for(summary_path)
    if artifact_root is None:
        return "unknown", "unknown"
    rel_parts = summary_path.relative_to(artifact_root).parts
    if len(rel_parts) >= 4 and rel_parts[0] in TRACK_LABELS:
        return rel_parts[0], rel_parts[1]
    if len(rel_parts) >= 3:
        return "unknown", rel_parts[0]
    return "unknown", "unknown"


def _bootstrap_ci(values: np.ndarray, *, n_samples: int, seed: int, ci_level: float, offset: float = 0.0) -> tuple[float, float]:
    values = np.asarray(values, dtype=np.float64)
    values = values[~np.isnan(values)]
    if values.size == 0:
        return math.nan, math.nan
    if n_samples <= 0:
        estimate = float(values.mean() - offset)
        return estimate, estimate

    rng = np.random.default_rng(seed)
    estimates = np.empty(n_samples, dtype=np.float64)
    for index in range(n_samples):
        sample_indices = rng.integers(0, values.shape[0], size=values.shape[0])
        estimates[index] = values[sample_indices].mean() - offset

    alpha = 1.0 - ci_level
    low, high = np.quantile(estimates, [alpha / 2.0, 1.0 - alpha / 2.0])
    return float(low), float(high)


def _per_user_seed_means(summary_path: Path, metric: str) -> np.ndarray | None:
    per_user_path = summary_path.parent / "per_user_metrics.csv"
    if not per_user_path.exists():
        return None
    per_user = pd.read_csv(per_user_path, usecols=["user_index", "eval_seed", metric])
    per_user[metric] = per_user[metric].astype(float)
    z_user = per_user.groupby("user_index", sort=True)[metric].mean()
    return z_user.to_numpy(dtype=np.float64)


def _relative_equivalence(summary_path: Path, metric: str, ours: float, paper: float | None) -> dict:
    if paper is None or paper == 0:
        return {
            "relative_margin_abs": math.nan,
            "relative_diff_pct": math.nan,
            "mean_within_relative_margin": None,
            "tost_diff_ci_low": math.nan,
            "tost_diff_ci_high": math.nan,
            "tost_equivalent_relative_margin": None,
            "tost_source": "missing paper value",
        }

    margin = abs(paper) * RELATIVE_EQ_MARGIN
    diff = ours - paper
    output = {
        "relative_margin_abs": margin,
        "relative_diff_pct": 100.0 * diff / paper,
        "mean_within_relative_margin": abs(diff) <= margin,
        "tost_diff_ci_low": math.nan,
        "tost_diff_ci_high": math.nan,
        "tost_equivalent_relative_margin": None,
        "tost_source": "missing per_user_metrics.csv",
    }

    z_user = _per_user_seed_means(summary_path, metric)
    if z_user is None:
        return output

    diff_low, diff_high = _bootstrap_ci(
        z_user,
        n_samples=BOOTSTRAP_SAMPLES,
        seed=BOOTSTRAP_SEED,
        ci_level=TOST_CI_LEVEL,
        offset=paper,
    )
    output.update(
        {
            "tost_diff_ci_low": diff_low,
            "tost_diff_ci_high": diff_high,
            "tost_equivalent_relative_margin": diff_low >= -margin and diff_high <= margin,
            "tost_source": "per_user_metrics.csv",
        }
    )
    return output


def load_eval_seed_runs(metric: str = REPORT_METRIC) -> pd.DataFrame:
    rows = []
    for artifact_root in ARTIFACT_ROOTS:
        if not artifact_root.exists():
            continue
        for summary_path in sorted(artifact_root.rglob("summary.json")):
            payload = json.loads(summary_path.read_text())
            eval_mode = payload.get("eval_mode", "eval_seeds" if len(payload.get("eval_seeds", [])) > 1 else "normal")
            if eval_mode != REQUIRED_EVAL_MODE:
                continue

            track, dataset_slug = _track_and_dataset(summary_path)
            dataset = DATASET_LABELS.get(payload.get("category"), DATASET_LABELS.get(dataset_slug, dataset_slug))
            try:
                metric_row = _metric_summary(payload, metric=metric)
            except KeyError:
                continue
            paper = _paper_value(dataset, metric)
            ours = float(metric_row["final_user_avg"])
            diff = None if paper is None else ours - paper
            eval_seed_std = float(metric_row.get("eval_seed_std", math.nan))
            diff_over_seed_std = math.nan
            if diff is not None and eval_seed_std > 0:
                diff_over_seed_std = abs(diff) / eval_seed_std

            ci_low = float(metric_row.get("user_bootstrap_ci_low", math.nan))
            ci_high = float(metric_row.get("user_bootstrap_ci_high", math.nan))
            paper_inside_ci = None
            if paper is not None and not math.isnan(ci_low) and not math.isnan(ci_high):
                paper_inside_ci = ci_low <= paper <= ci_high

            rel_eq = _relative_equivalence(summary_path, metric, ours, paper)

            rows.append(
                {
                    "artifact_root": str(artifact_root),
                    "track": track,
                    "config_source": TRACK_LABELS.get(track, track),
                    "dataset_slug": dataset_slug,
                    "dataset": dataset,
                    "session": summary_path.parent.name,
                    "summary_path": str(summary_path),
                    "checkpoint_path": payload.get("checkpoint_path"),
                    "eval_mode": eval_mode,
                    "n_eval_seeds": len(payload.get("eval_seeds", [])),
                    "n_users": int(metric_row.get("n_users", 0)),
                    "metric": metric,
                    "paper_ndcg10": paper,
                    "ours_ndcg10": ours,
                    "diff": diff,
                    "eval_seed_std": eval_seed_std,
                    "diff_over_seed_std": diff_over_seed_std,
                    "user_ci_level": float(metric_row.get("user_bootstrap_ci_level", math.nan)),
                    "user_ci_low": ci_low,
                    "user_ci_high": ci_high,
                    "paper_inside_user_ci": paper_inside_ci,
                    "mean_visited_items": _mean_visited_items(payload),
                    "bootstrap_samples_summary": int(payload.get("bootstrap_samples", 0)),
                    "bootstrap_seed_summary": int(payload.get("bootstrap_seed", 0)),
                    **rel_eq,
                }
            )

    return pd.DataFrame(rows)


def load_all_metric_runs(metrics: list[str] = TABLE_METRICS) -> pd.DataFrame:
    frames = [load_eval_seed_runs(metric) for metric in metrics]
    frames = [frame for frame in frames if not frame.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


all_runs = load_eval_seed_runs(REPORT_METRIC)
all_metric_runs = load_all_metric_runs(TABLE_METRICS)

if all_runs.empty:
    joined_roots = "\n".join(str(path) for path in ARTIFACT_ROOTS)
    print(f"No {REQUIRED_EVAL_MODE} summary.json files found under:\n{joined_roots}")
    latest = pd.DataFrame()
    latest_all_metrics = pd.DataFrame()
    latest_overview = pd.DataFrame()
else:
    latest = (
        all_runs.sort_values(["dataset", "track", "session"])
        .groupby(["dataset", "track"], as_index=False)
        .tail(1)
        .sort_values(["dataset", "track"])
        .reset_index(drop=True)
    )
    if all_metric_runs.empty:
        latest_all_metrics = pd.DataFrame()
    else:
        latest_sessions = latest[["dataset", "track", "session"]]
        latest_all_metrics = all_metric_runs.merge(latest_sessions, on=["dataset", "track", "session"], how="inner")
    latest_overview = latest[["dataset", "config_source", "session", "summary_path"]].copy()

latest_overview


,dataset,config_source,session,summary_path
0,Beauty,released README,20260611T202925005437Z_job23693178,/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval...
1,CDs,released README,20260612T172202266511Z_job23732814,/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval...
2,Sports,released README,20260611T202809147355Z_job23693177,/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval...
3,Toys,released README,20260611T202618380706Z_job23693175,/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval...


## Latest Runs

Useful for checking which seeded SASRec session is currently selected for each dataset/config source pair.


In [3]:
if latest_overview.empty:
    print(f"No {REQUIRED_EVAL_MODE} SASRec eval runs available.")
else:
    display(latest_overview)
    emit_latex_table("Latest runs", latest_overview)


,dataset,config_source,session,summary_path
0,Beauty,released README,20260611T202925005437Z_job23693178,/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval...
1,CDs,released README,20260612T172202266511Z_job23732814,/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval...
2,Sports,released README,20260611T202809147355Z_job23693177,/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval...
3,Toys,released README,20260611T202618380706Z_job23693175,/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval...



Latest runs LaTeX:

\begin{tabular}{lllll}
\toprule
 & dataset & config\_source & session & summary\_path \\
\midrule
0 & Beauty & released README & 20260611T202925005437Z\_job23693178 & /gpfs/home6/scur1202/RPG/artifacts/sasrec/eval\_seeds/released\_readme/beauty/20260611T202925005437Z\_job23693178/summary.json \\
1 & CDs & released README & 20260612T172202266511Z\_job23732814 & /gpfs/home6/scur1202/RPG/artifacts/sasrec/eval\_seeds/released\_readme/cds\_and\_vinyl/20260612T172202266511Z\_job23732814/summary.json \\
2 & Sports & released README & 20260611T202809147355Z\_job23693177 & /gpfs/home6/scur1202/RPG/artifacts/sasrec/eval\_seeds/released\_readme/sports\_and\_outdoors/20260611T202809147355Z\_job23693177/summary.json \\
3 & Toys & released README & 20260611T202618380706Z\_job23693175 & /gpfs/home6/scur1202/RPG/artifacts/sasrec/eval\_seeds/released\_readme/toys\_and\_games/20260611T202618380706Z\_job23693175/summary.json \\
\bottomrule
\end{tabular}



## Table 1: Our Results

This mirrors the RPG notebook's Table 1 layout. Values are `final_user_avg`: average each user over eval seeds first, then average over users.


In [4]:
if not all_metric_runs.empty and not latest_all_metrics.empty:
    dataset_order = ["Sports", "Beauty", "Toys", "CDs"]
    metric_order = TABLE_METRICS

    our_results = latest_all_metrics[
        ["config_source", "dataset", "metric", "ours_ndcg10"]
    ].copy()
    our_results["metric_label"] = our_results["metric"].map(METRIC_LABELS)
    our_results["Run"] = our_results["config_source"]

    paper_rows = []
    for dataset in dataset_order:
        for metric in metric_order:
            paper_value = PAPER_VALUES.get(metric, {}).get(dataset)
            if paper_value is None:
                continue
            paper_rows.append(
                {
                    "config_source": "paper",
                    "dataset": dataset,
                    "metric": metric,
                    "ours_ndcg10": paper_value,
                    "metric_label": METRIC_LABELS[metric],
                    "Run": "paper",
                }
            )
    if paper_rows:
        our_results = pd.concat([our_results, pd.DataFrame(paper_rows)], ignore_index=True)

    table_ours = our_results.pivot_table(
        index="Run",
        columns=["dataset", "metric_label"],
        values="ours_ndcg10",
        aggfunc="first",
    )

    ordered_columns = [
        (dataset, METRIC_LABELS[metric])
        for dataset in dataset_order
        for metric in metric_order
        if (dataset, METRIC_LABELS[metric]) in table_ours.columns
    ]
    ordered_index = [label for label in ["paper", "paper appendix", "released README"] if label in table_ours.index]
    table_ours = table_ours.loc[ordered_index, ordered_columns]
    table_ours.index.name = "Model/config"

    display(table_ours.style.format("{:.4f}", na_rep=""))
    emit_latex_table("Table 1", table_ours, formatter="{:.4f}", na_rep="", escape=True)
else:
    print("No seeded SASRec metric rows available for Table 1.")



Table 1 LaTeX:

\begin{tabular}{lllllllllllllllll}
\toprule
dataset & \multicolumn{4}{r}{Sports} & \multicolumn{4}{r}{Beauty} & \multicolumn{4}{r}{Toys} & \multicolumn{4}{r}{CDs} \\
metric_label & R@5 & N@5 & R@10 & N@10 & R@5 & N@5 & R@10 & N@10 & R@5 & N@5 & R@10 & N@10 & R@5 & N@5 & R@10 & N@10 \\
Model/config &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
paper & 0.0233 & 0.0154 & 0.0350 & 0.0192 & 0.0387 & 0.0249 & 0.0605 & 0.0318 & 0.0463 & 0.0306 & 0.0675 & 0.0374 & 0.0351 & 0.0177 & 0.0619 & 0.0263 \\
released README & 0.0192 & 0.0127 & 0.0312 & 0.0166 & 0.0396 & 0.0260 & 0.0590 & 0.0322 & 0.0461 & 0.0305 & 0.0668 & 0.0372 & 0.0317 & 0.0203 & 0.0511 & 0.0265 \\
\bottomrule
\end{tabular}



/scratch-local/63462/ipykernel_2395321/3248026414.py:88: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted = formatted.applymap(


## Table 2: NDCG@10 Paper Comparison

This keeps the formal comparison focused on the primary endpoint, `NDCG@10`.


In [5]:
if not latest.empty:
    table_reproduction = latest[
        [
            "dataset",
            "config_source",
            "paper_ndcg10",
            "ours_ndcg10",
            "diff",
            "relative_diff_pct",
            "relative_margin_abs",
            "tost_diff_ci_low",
            "tost_diff_ci_high",
            "mean_within_relative_margin",
            "tost_equivalent_relative_margin",
            "eval_seed_std",
            "diff_over_seed_std",
        ]
    ].copy()
    table_reproduction[f"{TOST_CI_LEVEL:.0%} TOST diff CI"] = table_reproduction.apply(
        lambda row: "missing" if pd.isna(row["tost_diff_ci_low"]) else f"[{row['tost_diff_ci_low']:+.5f}, {row['tost_diff_ci_high']:+.5f}]",
        axis=1,
    )
    table_reproduction = table_reproduction[
        [
            "dataset",
            "config_source",
            "paper_ndcg10",
            "ours_ndcg10",
            "diff",
            "relative_diff_pct",
            "relative_margin_abs",
            f"{TOST_CI_LEVEL:.0%} TOST diff CI",
            "mean_within_relative_margin",
            "tost_equivalent_relative_margin",
            "eval_seed_std",
            "diff_over_seed_std",
        ]
    ].rename(
        columns={
            "dataset": "Dataset",
            "config_source": "Config source",
            "paper_ndcg10": "Paper NDCG@10",
            "ours_ndcg10": "Ours NDCG@10",
            "diff": "Diff",
            "relative_diff_pct": "Rel. diff %",
            "relative_margin_abs": f"±{RELATIVE_EQ_MARGIN:.0%} margin",
            "mean_within_relative_margin": f"Mean within ±{RELATIVE_EQ_MARGIN:.0%}?",
            "tost_equivalent_relative_margin": f"TOST equivalent ±{RELATIVE_EQ_MARGIN:.0%}?",
            "eval_seed_std": "Eval seed std",
            "diff_over_seed_std": "|Diff| / seed std",
        }
    )
    table_reproduction_formatters = {
        "Paper NDCG@10": "{:.4f}",
        "Ours NDCG@10": "{:.5f}",
        "Diff": "{:+.5f}",
        "Rel. diff %": "{:+.1f}%",
        f"±{RELATIVE_EQ_MARGIN:.0%} margin": "{:.5f}",
        "Eval seed std": "{:.6f}",
        "|Diff| / seed std": "{:.1f}",
    }
    display(table_reproduction.style.format({
        "Paper NDCG@10": "{:.4f}",
        "Ours NDCG@10": "{:.5f}",
        "Diff": "{:+.5f}",
        "Rel. diff %": "{:+.1f}%",
        f"±{RELATIVE_EQ_MARGIN:.0%} margin": "{:.5f}",
        "Eval seed std": "{:.6f}",
        "|Diff| / seed std": "{:.1f}",
    }))
    emit_latex_table("Table 2", table_reproduction, column_formatters=table_reproduction_formatters)
else:
    print("No seeded SASRec NDCG@10 rows available for Table 2.")


,Dataset,Config source,Paper NDCG@10,Ours NDCG@10,Diff,Rel. diff %,±5% margin,90% TOST diff CI,Mean within ±5%?,TOST equivalent ±5%?,Eval seed std,|Diff| / seed std
0,Beauty,released README,0.0318,0.03219,+0.00039,+1.2%,0.00159,"[-0.00118, +0.00197]",True,False,0.000000,nan
1,CDs,released README,0.0263,0.02652,+0.00022,+0.8%,0.00132,"[-0.00054, +0.00097]",True,True,0.000000,nan
2,Sports,released README,0.0192,0.01659,-0.00261,-13.6%,0.00096,"[-0.00352, -0.00169]",False,False,0.000000,nan
3,Toys,released README,0.0374,0.03715,-0.00025,-0.7%,0.00187,"[-0.00208, +0.00156]",True,False,0.000000,nan



Table 2 LaTeX:

\begin{tabular}{lllllllllrrll}
\toprule
 & Dataset & Config source & Paper NDCG@10 & Ours NDCG@10 & Diff & Rel. diff \% & ±5\% margin & 90\% TOST diff CI & Mean within ±5\%? & TOST equivalent ±5\%? & Eval seed std & |Diff| / seed std \\
\midrule
0 & Beauty & released README & 0.0318 & 0.03219 & +0.00039 & +1.2\% & 0.00159 & [-0.00118, +0.00197] & True & False & 0.000000 &  \\
1 & CDs & released README & 0.0263 & 0.02652 & +0.00022 & +0.8\% & 0.00132 & [-0.00054, +0.00097] & True & True & 0.000000 &  \\
2 & Sports & released README & 0.0192 & 0.01659 & -0.00261 & -13.6\% & 0.00096 & [-0.00352, -0.00169] & False & False & 0.000000 &  \\
3 & Toys & released README & 0.0374 & 0.03715 & -0.00025 & -0.7\% & 0.00187 & [-0.00208, +0.00156] & True & False & 0.000000 &  \\
\bottomrule
\end{tabular}



## Table 3: Bootstrap CI

This reports user-level bootstrap uncertainty for `NDCG@10`, separated from eval-seed variance.


In [6]:
if not latest.empty:
    table_user_ci = latest[
        [
            "dataset",
            "config_source",
            "ours_ndcg10",
            "user_ci_low",
            "user_ci_high",
            "paper_inside_user_ci",
            "bootstrap_samples_summary",
            "bootstrap_seed_summary",
        ]
    ].copy()
    table_user_ci["User bootstrap CI"] = table_user_ci.apply(
        lambda row: f"[{row['user_ci_low']:.5f}, {row['user_ci_high']:.5f}]",
        axis=1,
    )
    table_user_ci = table_user_ci[
        ["dataset", "config_source", "ours_ndcg10", "User bootstrap CI", "paper_inside_user_ci", "bootstrap_samples_summary", "bootstrap_seed_summary"]
    ].rename(
        columns={
            "dataset": "Dataset",
            "config_source": "Config source",
            "ours_ndcg10": "Ours NDCG@10",
            "paper_inside_user_ci": "Paper inside user CI?",
            "bootstrap_samples_summary": "Summary bootstrap samples",
            "bootstrap_seed_summary": "Summary bootstrap seed",
        }
    )
    display(table_user_ci.style.format({"Ours NDCG@10": "{:.5f}"}))
    emit_latex_table("Table 3", table_user_ci, column_formatters={"Ours NDCG@10": "{:.5f}"})
else:
    print("No seeded SASRec NDCG@10 rows available for Table 3.")


,Dataset,Config source,Ours NDCG@10,User bootstrap CI,Paper inside user CI?,Summary bootstrap samples,Summary bootstrap seed
0,Beauty,released README,0.03219,"[0.03032, 0.03407]",True,5000,2024
1,CDs,released README,0.02652,"[0.02561, 0.02741]",True,5000,2024
2,Sports,released README,0.01659,"[0.01549, 0.01768]",False,5000,2024
3,Toys,released README,0.03715,"[0.03497, 0.03936]",True,5000,2024



Table 3 LaTeX:

\begin{tabular}{lllllrrr}
\toprule
 & Dataset & Config source & Ours NDCG@10 & User bootstrap CI & Paper inside user CI? & Summary bootstrap samples & Summary bootstrap seed \\
\midrule
0 & Beauty & released README & 0.03219 & [0.03032, 0.03407] & True & 5000 & 2024 \\
1 & CDs & released README & 0.02652 & [0.02561, 0.02741] & True & 5000 & 2024 \\
2 & Sports & released README & 0.01659 & [0.01549, 0.01768] & False & 5000 & 2024 \\
3 & Toys & released README & 0.03715 & [0.03497, 0.03936] & True & 5000 & 2024 \\
\bottomrule
\end{tabular}



## Appendix Table: Full Run Metadata

Useful for checking which checkpoint and summary file produced each row.


In [7]:
if not latest.empty:
    metadata_cols = [
        "dataset",
        "config_source",
        "session",
        "n_eval_seeds",
        "n_users",
        "mean_visited_items",
        "checkpoint_path",
        "summary_path",
    ]
    metadata_table = latest[metadata_cols].copy()
    display(metadata_table.style.format({"mean_visited_items": "{:.1f}"}))
    emit_latex_table("Appendix Table", metadata_table, column_formatters={"mean_visited_items": "{:.1f}"})
else:
    print("No seeded SASRec runs available for the appendix table.")


,dataset,config_source,session,n_eval_seeds,n_users,mean_visited_items,checkpoint_path,summary_path
0,Beauty,released README,20260611T202925005437Z_job23693178,10,22363,12101.0,/gpfs/work5/0/prjs2120/groups/group_16/artifacts/sasrec_modernized/ckpt/sasrec_modernized_beauty.pt,/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval_seeds/released_readme/beauty/20260611T202925005437Z_job23693178/summary.json
1,CDs,released README,20260612T172202266511Z_job23732814,10,75258,64443.0,/gpfs/work5/0/prjs2120/groups/group_16/artifacts/sasrec_modernized/ckpt/sasrec_modernized_cds_and_vinyl.pt,/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval_seeds/released_readme/cds_and_vinyl/20260612T172202266511Z_job23732814/summary.json
2,Sports,released README,20260611T202809147355Z_job23693177,10,35598,18357.0,/gpfs/work5/0/prjs2120/groups/group_16/artifacts/sasrec_modernized/ckpt/sasrec_modernized_sports_and_outdoors.pt,/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval_seeds/released_readme/sports_and_outdoors/20260611T202809147355Z_job23693177/summary.json
3,Toys,released README,20260611T202618380706Z_job23693175,10,19412,11924.0,/gpfs/work5/0/prjs2120/groups/group_16/artifacts/sasrec_modernized/ckpt/sasrec_modernized_toys_and_games.pt,/gpfs/home6/scur1202/RPG/artifacts/sasrec/eval_seeds/released_readme/toys_and_games/20260611T202618380706Z_job23693175/summary.json



Appendix Table LaTeX:

\begin{tabular}{llllrrlll}
\toprule
 & dataset & config\_source & session & n\_eval\_seeds & n\_users & mean\_visited\_items & checkpoint\_path & summary\_path \\
\midrule
0 & Beauty & released README & 20260611T202925005437Z\_job23693178 & 10 & 22363 & 12101.0 & /gpfs/work5/0/prjs2120/groups/group\_16/artifacts/sasrec\_modernized/ckpt/sasrec\_modernized\_beauty.pt & /gpfs/home6/scur1202/RPG/artifacts/sasrec/eval\_seeds/released\_readme/beauty/20260611T202925005437Z\_job23693178/summary.json \\
1 & CDs & released README & 20260612T172202266511Z\_job23732814 & 10 & 75258 & 64443.0 & /gpfs/work5/0/prjs2120/groups/group\_16/artifacts/sasrec\_modernized/ckpt/sasrec\_modernized\_cds\_and\_vinyl.pt & /gpfs/home6/scur1202/RPG/artifacts/sasrec/eval\_seeds/released\_readme/cds\_and\_vinyl/20260612T172202266511Z\_job23732814/summary.json \\
2 & Sports & released README & 20260611T202809147355Z\_job23693177 & 10 & 35598 & 18357.0 & /gpfs/work5/0/prjs2120/groups/group\_16/a